In [3]:

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA

# 1. Load dataset
df = pd.read_csv("/content/01 covid_toy.csv")

print("Data types:")
print(df.dtypes)

# 2. Display column names
print("\nNumerical columns:")
print(df.select_dtypes(include=np.number).columns)

print("\nCategorical columns:")
print(df.select_dtypes(include="object").columns)

display(df.head())

# 3. Replace '?' with missing values
df = df.replace("?", np.nan)

df["has_covid"] = df["has_covid"].astype("object")

print("\nMissing values:")
print(df.isnull().sum())

# 4. Convert age and fever into numeric values
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["fever"] = pd.to_numeric(df["fever"], errors="coerce")

# 5. Fill missing numerical values
numeric_features = df.select_dtypes(include=np.number)

imputer = SimpleImputer(strategy="mean")

numeric_features = pd.DataFrame(
    imputer.fit_transform(numeric_features),
    columns=numeric_features.columns,
    index=numeric_features.index
)

print("\nNumerical data after handling missing values:")
display(numeric_features.head())

# 6. Scale numerical features
scaler = MinMaxScaler().set_output(transform="pandas")

scaled_data = scaler.fit_transform(numeric_features)

print("\nScaled data:")
display(scaled_data.head())

# 7. Fill missing categorical values
categorical_features = df.select_dtypes(include=["object"])

for column in categorical_features.columns:
    if column != "has_covid":
        categorical_features[column] = categorical_features[column].fillna(
            categorical_features[column].mode().iloc[0]
            if not categorical_features[column].mode().empty
            else "Unknown"
        )

# 8. Encode categorical input features
categorical_input = categorical_features.drop(
    columns=["has_covid"]
)

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
).set_output(transform="pandas")

encoded_data = encoder.fit_transform(categorical_input)

print("\nEncoded categorical data:")
display(encoded_data.head())

# 9. Combine numerical and encoded categorical features
processed_features = pd.concat(
    [scaled_data, encoded_data],
    axis=1
)

print("\nProcessed features:")
display(processed_features.head())

# 10. Encode target column
target = (
    df["has_covid"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({"no": 0, "yes": 1})
)

# Remove rows with missing or invalid target values
valid_rows = target.notna()

processed_features = processed_features.loc[valid_rows]
target = target.loc[valid_rows].astype(int)

# 11. Select the best 3 features
selector = SelectKBest(
    score_func=f_classif,
    k=3
).set_output(transform="pandas")

selected_features = selector.fit_transform(
    processed_features,
    target
)

print("\nSelected features:")
display(selected_features.head())

# 12. Display feature selection scores
feature_scores = pd.DataFrame({
    "Feature": processed_features.columns,
    "Score": selector.scores_
}).sort_values(
    by="Score",
    ascending=False
)

print("\nFeature selection scores:")
display(feature_scores)

# 13. Standardize selected features for PCA
standard_scaler = StandardScaler().set_output(
    transform="pandas"
)

standardized_features = standard_scaler.fit_transform(
    selected_features
)

print("\nStandardized features:")
display(standardized_features.head())

# 14. Apply PCA
pca = PCA(n_components=2)

pca_result = pca.fit_transform(standardized_features)

pca_data = pd.DataFrame(
    pca_result,
    columns=["PC1", "PC2"],
    index=standardized_features.index
)

print("\nPCA results:")
display(pca_data.head())

# 15. Display explained variance
print("\nExplained variance ratio:")
print(pca.explained_variance_ratio_)

print("\nTotal explained variance:")
print(pca.explained_variance_ratio_.sum())


Data types:
age            int64
gender        object
fever        float64
cough         object
city          object
has_covid     object
dtype: object

Numerical columns:
Index(['age', 'fever'], dtype='object')

Categorical columns:
Index(['gender', 'cough', 'city', 'has_covid'], dtype='object')


,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No



Missing values:
age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

Numerical data after handling missing values:


,age,fever
0,60.0,103.0
1,27.0,100.0
2,42.0,101.0
3,31.0,98.0
4,65.0,101.0



Scaled data:


...


Standardized features:


,fever,cough_Mild,cough_Strong
0,1.111906,0.782881,-0.782881
1,-0.435592,0.782881,-0.782881
2,0.080241,0.782881,-0.782881
3,-1.467257,0.782881,-0.782881
4,0.080241,0.782881,-0.782881



PCA results:


,PC1,PC2
0,1.292370,0.889899
1,1.008449,-0.631330
2,1.103089,-0.124254
3,0.819168,-1.645483
4,1.103089,-0.124254



Explained variance ratio:
[0.6786972 0.3213028]

Total explained variance:
1.0
